<div align="center">

## **DS108 - TIỀN XỬ LÝ VÀ XÂY DỰNG BỘ DỮ LIỆU**

## **ĐỒ ÁN - THU THẬP & TIỀN XỬ LÝ DỮ LIỆU TÀI CHÍNH DOANH NGHIỆP CHO DỰ ĐOÁN LỢI NHUẬN**
</div>

**GVHD:** TS.Nguyễn Gia Tuấn Anh - CN: Trần Quốc Khánh

**Sinh viên thực hiện:**

| Họ và tên | MSSV |
|---|---|
| Võ Hạnh Nguyên | 24521218 |
| Phan Thị Ánh Nguyệt | 24521221 |

**Notebook:** `02_data_cleaning_and_imputation.ipynb` — Xử lý dữ liệu

---
### **Mục tiêu**
Notebook thực hiện quy trình tiền xử lý dữ liệu tài chính doanh nghiệp từ raw CSV thành feature matrix sẵn sàng cho ML: làm sạch, feature engineering, tạo biến mục tiêu, xử lý outlier, imputation và chuẩn hóa — với anti-leakage nghiêm ngặt ở mọi bước.

---
### ***Pipeline:***

1. Đọc dữ liệu
2. Làm sạch dữ liệu (Data cleaning)
3. Xây dựng đặc trưng (Feature Engineering)
4. Tạo biến mục tiêu (Target Variable Creation)
5. Xử lý outlier
6. Chia dữ liệu chuỗi thời gian (Time-Series Split)
7. Imputation
8. sklearn Preprocessing Pipeline (Winsorize + Scale)
9. Feature Selection
10. Pipeline Summary
---


## **Import & Logging configuration**

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
import re
import os
from typing import Tuple, List, Optional, Dict
from pathlib import Path

# sklearn — Pipeline & Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings('ignore')

# ── Logging configuration ─────────────────────────────────────
# Force reset handlers để tránh no-op trong Jupyter
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('DS108')

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Global constants ─────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent
FILE_PATH      = PROJECT_ROOT / 'data/raw/Data_DS108_raw.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/Data_DS108_processed.csv'
TRAIN_FS_PATH  = PROJECT_ROOT / 'data/processed/train_after_feature_selection.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
GROUP_COL      = 'symbol'
SECTOR_COL     = 'sector_group'           # dùng cho SectorWinsorizer
DATE_COL       = 'fiscaldateending'
TARGET_REG     = 'next_quarter_net_income'
TARGET_CLS     = 'profit_increase_next'
TRAIN_CUTOFF   = 2021   # năm <= cutoff → train | năm > cutoff → test (~75/25)
RANDOM_STATE   = 42
EXCHANGE_RATES = {
    'CNY': {
        'default': 0.14,
        2011: 0.155, 2012: 0.158, 2013: 0.162, 2014: 0.163, 2015: 0.159,
        2016: 0.150, 2017: 0.148, 2018: 0.151, 2019: 0.145, 2020: 0.145,
        2021: 0.155, 2022: 0.148, 2023: 0.141, 2024: 0.139, 2025: 0.138, 2026: 0.145
    },
    'TWD': {
        'default': 0.032,
        2006: 0.031, 2007: 0.030, 2008: 0.032, 2009: 0.030, 2010: 0.032,
        2011: 0.034, 2012: 0.034, 2013: 0.034, 2014: 0.033, 2015: 0.031,
        2016: 0.031, 2017: 0.033, 2018: 0.033, 2019: 0.032, 2020: 0.034,
        2021: 0.036, 2022: 0.034, 2023: 0.032, 2024: 0.031, 2025: 0.031, 2026: 0.032
    },
    'USD': {'default': 1.0}
}

---
## **Sector Mapping**

In [3]:
SYMBOL_SECTOR_MAP: Dict[str, str] = {
    # Technology
    'AAPL' : 'Technology',
    'MSFT' : 'Technology',
    'NVDA' : 'Technology',
    'AVGO' : 'Technology',
    'ORCL' : 'Technology',
    'TSM'  : 'Technology',
    'PLTR' : 'Technology',

    # Communication Services
    'META' : 'Communication Services',
    'GOOG' : 'Communication Services',
    'NFLX' : 'Communication Services',

    # Consumer Discretionary
    'AMZN' : 'Consumer Discretionary',
    'TSLA' : 'Consumer Discretionary',
    'HD'   : 'Consumer Discretionary',
    'BABA' : 'Consumer Discretionary',

    # Consumer Staples
    'COST' : 'Consumer Staples',
    'WMT'  : 'Consumer Staples',

    # Financials
    'JPM'  : 'Financials',
    'MA'   : 'Financials',
    'V'    : 'Financials',
    'BRK-B': 'Financials',

    # Healthcare
    'JNJ'  : 'Healthcare',
    'LLY'  : 'Healthcare',

    # Energy
    'XOM'  : 'Energy',
}

logger.info('Sector map loaded: %d symbols across %d sectors',
            len(SYMBOL_SECTOR_MAP),
            len(set(SYMBOL_SECTOR_MAP.values())))


20:08:50 | INFO | Sector map loaded: 23 symbols across 7 sectors


---
## **1. Đọc dữ liệu**


In [4]:
# Đọc dữ liệu
def load_data(filepath: str) -> pd.DataFrame:
    """Đọc CSV, lọc bỏ dòng rác (khuyết tiền tệ), log shape và sample."""
    try:
        # 1. Đọc file CSV
        df = pd.read_csv(filepath, low_memory=False)

        # 2. Loại bỏ các dòng bị rỗng cột reportedCurrency ngay từ đầu
        before_drop = df.shape[0]
        df = df.dropna(subset=['reportedCurrency'])
        dropped_count = before_drop - df.shape[0]

        # 3. Ghi log để theo dõi số lượng dòng đã loại bỏ
        if dropped_count > 0:
            logger.info('Đã loại bỏ %d dòng rác bị khuyết reportedCurrency.', dropped_count)

        logger.info('Loaded "%s" → shape sau khi lọc rác %s', filepath, df.shape)
        return df

    except FileNotFoundError:
        logger.error('File not found: %s', filepath)
        raise

df_raw = load_data(FILE_PATH)

20:08:50 | INFO | Đã loại bỏ 1 dòng rác bị khuyết reportedCurrency.
20:08:50 | INFO | Loaded "d:\DS108\ds108\data\raw\Data_DS108_raw.csv" → shape sau khi lọc rác (1761, 27)


---

## **2. Làm sạch dữ liệu (Data Cleaning)**

In [5]:
# Xử lí duplicate theo Business Logic
def detect_business_duplicates(
    df: pd.DataFrame,
    key_cols: List[str] = ['symbol', 'fiscaldateending'],
) -> pd.DataFrame:
    """
    Phát hiện & log báo cáo các bản ghi trùng lặp.
    Không xoá — chỉ log để quan sát. Việc xoá do resolve_business_duplicates().

    Returns: DataFrame chứa các dòng trùng (để debug).
    """
    available_keys = [c for c in key_cols if c in df.columns]
    if len(available_keys) < 2:
        logger.warning('[DupDetect] Không đủ cột khoá %s trong df. Có: %s',
                       key_cols, list(df.columns))
        return pd.DataFrame()

    dup_mask   = df.duplicated(subset=available_keys, keep=False)
    df_dups    = df[dup_mask].copy()
    n_dup_rows = dup_mask.sum()
    n_dup_grps = df_dups.groupby(available_keys).ngroups if n_dup_rows > 0 else 0

    logger.info('[DupDetect] Phát hiện %d dòng trùng, thuộc %d cặp (symbol, kỳ báo cáo)',
                n_dup_rows, n_dup_grps)
    if n_dup_rows > 0:
        sample = (df_dups.groupby(available_keys).size()
                         .reset_index(name='count')
                         .sort_values('count', ascending=False)
                         .head(10))
        logger.info('[DupDetect] Top nhóm trùng nhiều nhất:\n%s',
                    sample.to_string(index=False))
    return df_dups


def resolve_business_duplicates(
    df: pd.DataFrame,
    key_cols: List[str]     = ['symbol', 'fiscaldateending'],
    date_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Xử lý trùng lặp theo Business Logic — 3 bước:

    Bước 1: Tính _nan_count = số NaN mỗi dòng.
    Bước 2: Sort tăng dần theo _nan_count (ít NaN = đầy đủ nhất lên trước)
            + giảm dần theo date_col nếu có (bản đính chính mới nhất lên trước).
            Cột date_col được tự suy ra từ các tên phổ biến nếu không truyền vào.
    Bước 3: drop_duplicates(keep='first') → giữ bản ghi chất lượng nhất.

    Log rõ số dòng đã xoá.
    """
    df       = df.copy()
    n_before = len(df)
    available_keys = [c for c in key_cols if c in df.columns]

    if len(available_keys) < 2:
        logger.warning('[DupResolve] Không đủ cột khoá — bỏ qua.')
        return df
    if df.duplicated(subset=available_keys, keep=False).sum() == 0:
        logger.info('[DupResolve] Không có bản ghi trùng. Bỏ qua.')
        return df

    # Bước 1: Đếm NaN mỗi dòng
    df['_nan_count'] = df.isnull().sum(axis=1)

    # Bước 2: Tự suy ra cột ngày thu thập nếu không truyền
    _candidates = ['reporteddate', 'reported_date', 'collecteddate',
                   'collected_date', 'updatedate', 'update_date', 'date_collected']
    if date_col is None:
        for cand in _candidates:
            if cand in df.columns:
                date_col = cand
                logger.info('[DupResolve] Tự suy ra cột ngày: "%s"', date_col)
                break

    sort_keys      = available_keys + ['_nan_count']
    sort_ascending = [True, True, True]   # asc cho 2 key + asc cho nan_count

    if date_col and date_col in df.columns:
        df[date_col]   = pd.to_datetime(df[date_col], errors='coerce')
        sort_keys      = available_keys + ['_nan_count', date_col]
        sort_ascending = [True, True, True, False]   # desc date → mới nhất lên trước
        logger.info('[DupResolve] Sort: %s | ascending=%s', sort_keys, sort_ascending)
    else:
        logger.info('[DupResolve] Không có cột ngày → chỉ sort theo nan_count.')

    # Bước 3: Giữ bản chất lượng nhất
    df = (df.sort_values(sort_keys, ascending=sort_ascending)
            .reset_index(drop=True)
            .drop_duplicates(subset=available_keys, keep='first')
            .reset_index(drop=True))
    df.drop(columns=['_nan_count'], inplace=True, errors='ignore')

    n_removed = n_before - len(df)
    logger.info('[DupResolve] %d → %d dòng | Đã xoá %d dòng trùng lặp',
                n_before, len(df), n_removed)
    return df


In [6]:
# Làm sạch dữ liệu
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pipeline làm sạch:
      1. Chuẩn hóa tên cột
      2. Xử lý trùng lặp theo Business Logic (detect → resolve)
      3. Thay invalid strings → NaN
      4. Parse fiscaldateending → datetime
      5. Ép kiểu numeric → float64
      6. Loại cột missing > 50%
      7. Thêm sector_group
      8. Quy đổi tiền tệ sang USD theo tỷ giá động (năm + loại tiền)
    """
    df = df.copy()
    n0 = len(df)

    # ── 1. Chuẩn hóa tên cột ──────────────────────────
    # (normalize trước để detect/resolve dùng đúng tên cột lowercase)
    df.columns = [
        re.sub(r'[^a-z0-9]+', '_', col.lower()).strip('_')
        for col in df.columns
    ]
    logger.info('[Clean] Columns normalized: %s', list(df.columns))

    # ── 2. Duplicate theo Business Logic ──────────────────────
    # Thay thế drop_duplicates() đơn giản:
    # Phát hiện → sắp xếp theo (ít NaN nhất, ngày mới nhất) → giữ bản tốt nhất
    _ = detect_business_duplicates(df)      # log & báo cáo
    df = resolve_business_duplicates(df)    # giữ bản ghi chất lượng nhất

    # ── 3. Invalid values → NaN ───────────────────────────────
    INVALID = ['None', 'none', 'null', 'NULL', 'N/A', 'n/a', '--', '', 'nan']
    df.replace(INVALID, np.nan, inplace=True)

    # ── 4. Parse date ─────────────────────────────────────────
    if 'fiscaldateending' in df.columns:
        df['fiscaldateending'] = pd.to_datetime(df['fiscaldateending'], errors='coerce')
        n_nat = df['fiscaldateending'].isna().sum()
        logger.info('[Clean] fiscaldateending parsed. NaT count: %d', n_nat)

    # ── 5. Ép kiểu numeric ────────────────────────────────────
    NON_NUMERIC = {'symbol', 'fiscaldateending', 'reportedcurrency'}
    numeric_cols = [c for c in df.columns if c not in NON_NUMERIC]
    converted = 0
    for col in numeric_cols:
        converted_col = pd.to_numeric(df[col], errors='coerce')
        if converted_col.notna().sum() > 0:
            df[col] = converted_col.astype('float64')
            converted += 1
    logger.info('[Clean] Converted %d columns to float64', converted)

    # ── 6. Drop high-missing columns (> 50%) ─────────────────
    # Lưu ý anti-leakage: missing_pct được tính trên TOÀN BỘ data thay vì chỉ
    # train. Đây là quyết định CỐ Ý vì: (a) việc một cột có > 50% missing
    # thường là đặc điểm cấu trúc của nguồn dữ liệu (không phải pattern học
    # được từ tập train), (b) nếu tính riêng trên train có thể loại bỏ cột
    # giá trị, tạo sự bất đối xứng schema train/test gây khó debug. Các bước
    # học tham số (median, quantile, scaler) phía sau vẫn fit() chỉ trên train.
    PROTECT = {'symbol', 'fiscaldateending', 'reportedcurrency',
               'totalrevenue', 'netincome', 'grossprofit'}
    missing_pct = df.isnull().mean()
    drop_cols = [
        c for c in df.columns
        if missing_pct[c] > 0.50 and c not in PROTECT
    ]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)
        logger.warning('[Clean] Dropped %d high-missing columns: %s', len(drop_cols), drop_cols)

    # ── 7. Sector group ─────────────────────────────────────
    if 'symbol' in df.columns:
        df['sector_group'] = (
            df['symbol'].map(SYMBOL_SECTOR_MAP).fillna('Other')
        )
        unmapped = df[df['sector_group'] == 'Other']['symbol'].unique()
        if len(unmapped):
            logger.warning('[Clean] %d unmapped symbols → "Other": %s',
                           len(unmapped), list(unmapped))

    # ── 8. Quy đổi tiền tệ sang USD ─────────────────────────────────────
    if 'reportedcurrency' in df.columns:
        df['reportedcurrency'] = df['reportedcurrency'].fillna('USD').str.upper()

        # TẠO CỘT NĂM ĐỘNG: Trích xuất năm từ cột fiscaldateending đã parse ở bước 4
        if 'fiscaldateending' in df.columns:
            # .dt.year sẽ lấy ra số năm (ví dụ: 2021), nếu lỗi điền tạm là 0
            df['derived_year'] = df['fiscaldateending'].dt.year.fillna(0).astype(int)
            year_col = 'derived_year'
        else:
            # Phòng hờ nếu không thấy cột date, tìm tiếp fyear hoặc year gốc
            year_col = 'year' if 'year' in df.columns else ('fyear' if 'fyear' in df.columns else None)

        if year_col is not None:
            if year_col != 'derived_year':
                df[year_col] = df[year_col].fillna(0).astype(int)

            # Hàm nội bộ tra cứu tỷ giá động
            def map_rate(row):
                    curr      = row['reportedcurrency']
                    yr        = int(row[year_col])
                    curr_dict = EXCHANGE_RATES.get(curr, {'default': 1.0})

                    if curr != 'USD' and yr not in curr_dict:
                        logger.warning(
                            '[Currency] Năm %d không có trong bảng tỷ giá '
                            'cho %s → dùng default rate %.4f',
                            yr, curr, curr_dict['default']
                        )
                    return curr_dict.get(yr, curr_dict['default'])

            rates = df.apply(map_rate, axis=1)

            # Xóa cột tạm sau khi tính xong để tránh làm bẩn dữ liệu huấn luyện
            if 'derived_year' in df.columns:
                df.drop(columns=['derived_year'], inplace=True)
        else:
            logger.warning('No time column found. Using default exchange rates.')
            rates = df['reportedcurrency'].map(
                    lambda x: EXCHANGE_RATES.get(x, {'default': 1.0})['default']
                )

        # Danh sách các cột chứa mệnh giá tuyệt đối cần được quy đổi sang USD
        monetary_cols = [
            'totalrevenue', 'grossprofit', 'costofrevenue', 'costofgoodsandservicessold',
            'operatingincome', 'sellinggeneralandadministrative', 'researchanddevelopment',
            'operatingexpenses', 'investmentincomenet', 'netinterestincome', 'interestincome',
            'interestexpense', 'noninterestincome', 'othernonoperatingincome', 'depreciation',
            'depreciationandamortization', 'incomebeforetax', 'incometaxexpense',
            'interestanddebtexpense', 'netincomefromcontinuingoperations',
            'comprehensiveincomenetoftax', 'ebit', 'ebitda', 'netincome']

        # Thực hiện nhân quy đổi hàng loạt
        for col in monetary_cols:
            if col in df.columns:
                df[col] = df[col] * rates

        logger.info("Successfully converted monetary columns into USD using provided annual exchange rates.")

        # Vẫn giữ lại flag 'is_usd' (1 nếu gốc là USD, 0 nếu là CNY/TWD)
        df['is_usd'] = (df['reportedcurrency'] == 'USD').astype(int)

    return df

# ── Run ───────────────────────────────────────────────────────
df_clean = clean_data(df_raw)

# Quick EDA
print('\n=== Missing values (top 10) ===')
missing = (df_clean.isnull().mean() * 100).sort_values(ascending=False)
display(missing[missing > 0].head(10).rename('% missing').to_frame())
print(f'\nSymbols ({df_clean["symbol"].nunique()}): {sorted(df_clean["symbol"].unique())}')
print(f'Date range: {df_clean["fiscaldateending"].min().date()} → {df_clean["fiscaldateending"].max().date()}')


20:08:50 | INFO | [Clean] Columns normalized: ['symbol', 'fiscaldateending', 'reportedcurrency', 'totalrevenue', 'costofrevenue', 'costofgoodsandservicessold', 'grossprofit', 'sellinggeneralandadministrative', 'researchanddevelopment', 'operatingexpenses', 'operatingincome', 'investmentincomenet', 'netinterestincome', 'interestincome', 'interestexpense', 'noninterestincome', 'othernonoperatingincome', 'depreciation', 'depreciationandamortization', 'incomebeforetax', 'incometaxexpense', 'interestanddebtexpense', 'netincomefromcontinuingoperations', 'comprehensiveincomenetoftax', 'ebit', 'ebitda', 'netincome']
20:08:50 | INFO | [DupDetect] Phát hiện 0 dòng trùng, thuộc 0 cặp (symbol, kỳ báo cáo)
20:08:50 | INFO | [DupResolve] Không có bản ghi trùng. Bỏ qua.
20:08:50 | INFO | [Clean] fiscaldateending parsed. NaT count: 0
20:08:50 | INFO | [Clean] Converted 19 columns to float64
20:08:50 | WARNING | [Clean] Dropped 8 high-missing columns: ['investmentincomenet', 'netinterestincome', 'inter


=== Missing values (top 10) ===


,% missing
researchanddevelopment,18.7394
netincomefromcontinuingoperations,9.0857
sellinggeneralandadministrative,3.5775
depreciationandamortization,1.7036
interestexpense,1.3061
ebitda,0.9654
ebit,0.4543
netincome,0.2271
incomebeforetax,0.2271
grossprofit,0.2271



Symbols (23): ['AAPL', 'AMZN', 'AVGO', 'BABA', 'BRK-B', 'COST', 'GOOG', 'HD', 'JNJ', 'JPM', 'LLY', 'MA', 'META', 'MSFT', 'NFLX', 'NVDA', 'ORCL', 'PLTR', 'TSLA', 'TSM', 'V', 'WMT', 'XOM']
Date range: 2006-01-31 → 2026-03-31


---
## **3. Xây dựng đặc trưng (Feature Engineering)**

In [7]:
def safe_divide(num: pd.Series, den: pd.Series) -> pd.Series:
    safe_den = den.mask(den.abs() <= 1e-9)   # den ≈ 0 → NaN
    return num / safe_den                     # NaN / x = NaN, không raise warning


def _add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo year, quarter, quarter_label từ fiscaldateending."""
    df['year']          = df[DATE_COL].dt.year
    df['quarter']       = df[DATE_COL].dt.quarter          # 1-4
    df['quarter_label'] = 'Q' + df['quarter'].astype(str)  # Q1-Q4
    return df


def _add_margin_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Margin ratios (không có leakage — chỉ dùng cùng quý):
      gross_margin, operating_margin, net_margin, ebitda_margin
    """
    rev = df['totalrevenue']
    pairs = [
        ('gross_margin',     'grossprofit'),
        ('operating_margin', 'operatingincome'),
        ('net_margin',       'netincome'),
        ('ebitda_margin',    'ebitda'),
    ]
    for new_col, src in pairs:
        if src in df.columns:
            df[new_col] = safe_divide(df[src], rev)
    return df


def _add_qoq_growth(df: pd.DataFrame) -> pd.DataFrame:
    """
    QoQ growth (shift 1 quý trong cùng company):
      revenue_qoq, netincome_qoq, grossprofit_qoq

    Clip [-1, 5] = winsorize hardcode ở mức kinh tế hợp lý:
      -1  = -100% (mất hoàn toàn doanh thu/lợi nhuận)
      +5  = +500% (tăng gấp 6 lần — outlier kinh tế cực hiếm)
    Ngưỡng cứng, không học từ data → không gây leakage.
    Mục đích: chặn divide-by-near-zero blow-up khi mẫu số nhỏ.
    """
    for src, name in [('totalrevenue', 'revenue_qoq'),
                      ('netincome',    'netincome_qoq'),
                      ('grossprofit',  'grossprofit_qoq')]:
        if src not in df.columns:
            continue
        prev = df.groupby(GROUP_COL)[src].shift(1)
        df[name] = safe_divide(df[src] - prev, prev.abs()).clip(-1, 5)
    return df


def _add_yoy_growth(df: pd.DataFrame) -> pd.DataFrame:
    """
    YoY growth (shift 4 quý — cùng quý năm trước):
      revenue_yoy_growth, income_yoy_growth, grossprofit_yoy_growth

    Cùng lý do clip [-1, 5] như QoQ — xem _add_qoq_growth().
    """
    for src, name in [('totalrevenue', 'revenue_yoy_growth'),
                      ('netincome',    'income_yoy_growth'),
                      ('grossprofit',  'grossprofit_yoy_growth')]:
        if src not in df.columns:
            continue
        prev_yoy = df.groupby(GROUP_COL)[src].shift(4)
        df[name] = safe_divide(df[src] - prev_yoy, prev_yoy.abs()).clip(-1, 5)
    return df


# ── Rolling helper (tránh bug lambda closure pandas >= 2.0) ──
def _rolling_stat(series: pd.Series,
                  group_key: pd.Series,
                  window: int,
                  stat: str,
                  min_periods: int = 1) -> pd.Series:
    """
    Tính rolling statistic an toàn theo group.
    Dùng named function thay lambda để tránh lỗi
    'float() argument must be ... not Series' trên pandas >= 2.0.
    """
    def _apply(g):
        r = g.rolling(window, min_periods=min_periods)
        if stat == 'mean': return r.mean()
        if stat == 'std':  return r.std()
        if stat == 'max':  return r.max()
        if stat == 'min':  return r.min()
        return r.mean()
    return series.groupby(group_key).transform(_apply)


def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rolling statistics (window 4q và 8q).
    Anti-leakage: shift(1) TRƯỚC khi rolling → không dùng giá trị hiện tại.

    min_periods quy ước:
      - mean / max / min: min_periods = window // 2 (4q→2, 8q→4) hoặc 1
        → cho phép có giá trị sớm khi company mới có ít quý lịch sử.
      - std:              min_periods = 2 (BẮT BUỘC) — std với 1 mẫu = NaN
        về mặt toán học, không có ý nghĩa thống kê.

    Features tạo ra:
      *_rolling_mean_4q, *_rolling_std_4q
      *_rolling_max_4q,  *_rolling_min_4q
      *_rolling_mean_8q, *_rolling_std_8q
    """
    roll_sources = [
        ('totalrevenue', 'revenue'),
        ('netincome',    'income'),
        ('net_margin',   'net_margin'),
        ('gross_margin', 'gross_margin'),
        ('revenue_qoq',  'revenue_qoq'),
    ]

    for src, prefix in roll_sources:
        if src not in df.columns:
            continue

        # shift(1) theo group: rolling KHÔNG chứa giá trị quý hiện tại
        shifted = df.groupby(GROUP_COL)[src].shift(1)
        grp     = df[GROUP_COL]

        for w, wname in [(4, '4q'), (8, '8q')]:
            mp = max(1, w // 2)   # min_periods cho mean
            df[f'{prefix}_rolling_mean_{wname}'] = _rolling_stat(shifted, grp, w, 'mean', mp)
            df[f'{prefix}_rolling_std_{wname}']  = _rolling_stat(shifted, grp, w, 'std',  2)  # std cần ≥ 2 mẫu

        # max / min chỉ tính window 4q
        df[f'{prefix}_rolling_max_4q'] = _rolling_stat(shifted, grp, 4, 'max', 1)
        df[f'{prefix}_rolling_min_4q'] = _rolling_stat(shifted, grp, 4, 'min', 1)

    return df


def _add_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Momentum features — đo xu hướng tăng tốc/giảm tốc:
      revenue_momentum  = QoQ_t - QoQ_{t-1}
      earnings_momentum = netincome_qoq_t - netincome_qoq_{t-1}
    """
    for src, name in [('revenue_qoq',  'revenue_momentum'),
                      ('netincome_qoq','earnings_momentum')]:
        if src not in df.columns:
            continue
        prev = df.groupby(GROUP_COL)[src].shift(1)
        df[name] = df[src] - prev
    return df


def _add_growth_stability(df: pd.DataFrame) -> pd.DataFrame:
    """
    Growth volatility (rolling std của QoQ growth):
      revenue_growth_volatility_4q
      income_growth_volatility_4q

    Anti-leakage: shift(1) trước rolling.
    min_periods=2 vì std cần ít nhất 2 quan sát.
    """
    for src, name in [('revenue_qoq',  'revenue_growth_volatility_4q'),
                      ('netincome_qoq','income_growth_volatility_4q')]:
        if src not in df.columns:
            continue
        shifted = df.groupby(GROUP_COL)[src].shift(1)
        df[name] = _rolling_stat(shifted, df[GROUP_COL], 4, 'std', 2)
    return df


def _add_lagged_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lagged features (1, 2, 4 quý trước) cho các cột quan trọng.
    Anti-leakage: shift(n) đảm bảo chỉ dùng thông tin quá khứ.
    """
    lag_sources = [
        'totalrevenue', 'netincome', 'grossprofit',
        'net_margin', 'gross_margin', 'operating_margin',
        'revenue_qoq', 'netincome_qoq',
    ]
    count = 0
    for src in lag_sources:
        if src not in df.columns:
            continue
        for lag in [1, 2, 4]:
            df[f'{src}_lag{lag}'] = df.groupby(GROUP_COL)[src].shift(lag)
            count += 1
    logger.info('[FE] Created %d lagged features', count)
    return df


def _check_time_gaps(df: pd.DataFrame, threshold_days: int = 100) -> None:
    """
    Cảnh báo nếu có gap > threshold_days trong chuỗi quý của bất kỳ symbol.

    Lý do (xem EDA notebook 03, Section 4.7):
      Dữ liệu hàng quý đáng lẽ liên tục ~90 ngày/quý. Nếu có gap > 100 ngày
      tức là CÓ ÍT NHẤT 1 QUÝ BỊ THIẾU giữa chừng. Khi đó:
        - shift(4) sẽ KHÔNG còn lấy đúng "cùng quý năm trước" → YoY growth sai
        - shift(1) cũng không lấy đúng quý liền trước → QoQ growth sai
        - rolling() vẫn chạy đúng theo thứ tự nhưng cửa sổ thời gian không đều

    Hàm này chỉ LOG WARNING (không sửa data) để người dùng nắm rõ
    và xử lý phù hợp (interpolate, hoặc loại bỏ symbol có gap nghiêm trọng).
    """
    issues = []
    for sym, grp in df.groupby(GROUP_COL):
        dates = grp[DATE_COL].sort_values().dropna()
        if len(dates) < 2:
            continue
        diffs = dates.diff().dt.days.dropna()
        n_bad = int((diffs > threshold_days).sum())
        if n_bad > 0:
            issues.append((sym, n_bad, int(diffs.max())))

    if issues:
        logger.warning('[FE] Time-series gaps phát hiện ở %d/%d symbol — '
                       'shift(4)/YoY có thể bị lệch:', len(issues), df[GROUP_COL].nunique())
        for sym, n, max_gap in issues:
            logger.warning('   %s: %d gap > %d ngày | max gap = %d ngày',
                           sym, n, threshold_days, max_gap)
    else:
        logger.info('[FE] Time-series check: tất cả symbol có chuỗi quý liên tục '
                    '(≤ %d ngày giữa các quý).', threshold_days)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pipeline feature engineering tổng hợp.
    Thứ tự quan trọng: sort by symbol+date trước khi shift/rolling.
    """
    logger.info('[FE] Starting feature engineering on shape %s', df.shape)

    df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)

    # Validate chuỗi thời gian trước khi tính shift/rolling
    # (xem EDA notebook 03, Section 4.7 — Time-Series Gap Check)
    _check_time_gaps(df)

    df = _add_date_features(df)
    df = _add_margin_features(df)
    df = _add_qoq_growth(df)
    df = _add_yoy_growth(df)
    df = _add_rolling_features(df)
    df = _add_momentum_features(df)
    df = _add_growth_stability(df)
    df = _add_lagged_features(df)

    logger.info('[FE] Done. Total columns: %d', df.shape[1])
    return df


df_fe = engineer_features(df_clean.copy())

fe_cols = [c for c in df_fe.columns if c not in df_clean.columns]
logger.info('[FE] New features created (%d): %s', len(fe_cols), fe_cols)


20:08:50 | INFO | [FE] Starting feature engineering on shape (1761, 21)
20:08:50 | WARNING | [FE] Time-series gaps phát hiện ở 1/23 symbol — shift(4)/YoY có thể bị lệch:
20:08:50 | WARNING |    TSLA: 2 gap > 100 ngày | max gap = 275 ngày
20:08:50 | INFO | [FE] Created 24 lagged features
20:08:50 | INFO | [FE] Done. Total columns: 92
20:08:50 | INFO | [FE] New features created (71): ['year', 'quarter', 'quarter_label', 'gross_margin', 'operating_margin', 'net_margin', 'ebitda_margin', 'revenue_qoq', 'netincome_qoq', 'grossprofit_qoq', 'revenue_yoy_growth', 'income_yoy_growth', 'grossprofit_yoy_growth', 'revenue_rolling_mean_4q', 'revenue_rolling_std_4q', 'revenue_rolling_mean_8q', 'revenue_rolling_std_8q', 'revenue_rolling_max_4q', 'revenue_rolling_min_4q', 'income_rolling_mean_4q', 'income_rolling_std_4q', 'income_rolling_mean_8q', 'income_rolling_std_8q', 'income_rolling_max_4q', 'income_rolling_min_4q', 'net_margin_rolling_mean_4q', 'net_margin_rolling_std_4q', 'net_margin_rolling_me

---
## **4. Tạo biến mục tiêu (Target Variable Creation)**

In [8]:
def create_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo 2 biến mục tiêu dự đoán quý TIẾP THEO:

    1. next_quarter_net_income  → Regression
    2. profit_increase_next     → Classification (binary)
        1 = quý tới TĂNG so với quý hiện tại
        0 = quý tới GIẢM hoặc BẰNG quý hiện tại
        (Không tăng được coi là "không có profit increase" → label 0.)

    Anti-leakage: shift(-1) chỉ dùng để TẠO TARGET, không nằm trong features.

    Edge case:
      - Dòng cuối cùng của mỗi symbol có next_income = NaN
        → cả 2 target = NaN, sẽ bị loại ở split_time_series()
          qua filter df[target_col].notna().
    """
    df = df.copy()

    # next_income xuất hiện 2 lần (cho REG và CLS) — tính 1 lần dùng chung
    next_income = df.groupby(GROUP_COL)['netincome'].shift(-1)

    # 1. Regression target
    df[TARGET_REG] = next_income

    # 2. Classification target
    #    FIX (cũ): khi next == current gán NaN → mất ~1-2% dòng không cần thiết.
    #    Sửa: gán 0 cho cả "giảm" và "bằng" — chỉ NaN khi next_income = NaN
    #    (dòng cuối symbol).
    df[TARGET_CLS] = (next_income > df['netincome']).astype(float)
    df.loc[next_income.isna(), TARGET_CLS] = np.nan

    # Log
    valid_reg = df[TARGET_REG].notna().sum()
    valid_cls = df[TARGET_CLS].notna().sum()
    inc = (df[TARGET_CLS] == 1).sum()
    dec = (df[TARGET_CLS] == 0).sum()

    logger.info('[Target] Regression valid rows : %d', valid_reg)
    logger.info('[Target] Classification valid  : %d (↑%d %.1f%% | ↓/=%d %.1f%%)',
                valid_cls, inc, inc/valid_cls*100, dec, dec/valid_cls*100)
    return df


df_fe = create_target(df_fe)


20:08:50 | INFO | [Target] Regression valid rows : 1735
20:08:50 | INFO | [Target] Classification valid  : 1735 (↑964 55.6% | ↓/=771 44.4%)


---
## **5. Xử lí Outlier (Outlier Handling)**

Chia outlier handling thành 2 lớp tương ứng với 2 nhóm biến có bản chất khác nhau:

**Lớp A — Biến quy mô tuyệt đối** (`totalrevenue`, `netincome`, `ebit`...):
- Phương pháp: **Sign-Log Transform** — `f(x) = sign(x) · log(1 + |x|)`
- Giữ nguyên DẤU (lãi/lỗ), nén scale mạnh, monotone, không tham số.
- *Không cần `fit()` trên train* → là hàm **tất định**, áp được trên toàn bộ data trước split mà **không gây leakage**.
- Bước này được thực hiện ở **Section 5 (cell này)**.

**Lớp B — Biến tỷ lệ / tăng trưởng** (`net_margin`, `revenue_yoy_growth`, ...):
- Phương pháp: **Sector Winsorization** — clip về phân vị [1%, 99%] theo từng nhóm ngành.
- *Phải `fit()` chỉ trên train* để học ngưỡng phân vị → test dùng ngưỡng của train để clip.
- Bước này được thực hiện ở **Section 8 (sklearn Pipeline)**, sau khi đã split.

> ⚠️  **Anti-leakage chỉ áp dụng cho Lớp B (Winsorize)** vì có học tham số (quantile).
> Lớp A (Sign-Log) là hàm cố định, không có tham số học được → an toàn trước split.


In [9]:
# ── Danh sách cột theo nhóm ──────────────────────────────────

# Nhóm A: Biến quy mô tuyệt đối (phân phối lệch phải, có thể âm)
SCALE_COLS = [
    'totalrevenue', 'grossprofit', 'operatingincome',
    'netincome', 'ebit', 'ebitda',
    'costofrevenue', 'costofgoodsandservicessold',
    'operatingexpenses', 'incomebeforetax', 'incometaxexpense',
    'sellinggeneralandadministrative', 'researchanddevelopment',
    'depreciationandamortization',
    # Lagged versions (tạo ở Section 3)
    'totalrevenue_lag1', 'totalrevenue_lag2', 'totalrevenue_lag4',
    'netincome_lag1',    'netincome_lag2',    'netincome_lag4',
    'grossprofit_lag1',  'grossprofit_lag2',  'grossprofit_lag4',
]

LOG1P_COLS = SCALE_COLS

# Nhóm B: Biến tỷ lệ / tăng trưởng → winsorize [1%–99%]
# Lưu ý: bước winsorize THỰC HIỆN Ở SECTION 8 (sau split), không phải ở đây.
RATIO_COLS_WINSOR = [
    'gross_margin', 'operating_margin', 'net_margin', 'ebitda_margin',
    'revenue_qoq', 'netincome_qoq', 'grossprofit_qoq',
    'revenue_yoy_growth', 'income_yoy_growth', 'grossprofit_yoy_growth',
    'revenue_momentum', 'earnings_momentum',
    'revenue_growth_volatility_4q', 'income_growth_volatility_4q',
    'net_margin_lag1',     'net_margin_lag2',     'net_margin_lag4',
    'revenue_qoq_lag1',    'revenue_qoq_lag2',    'revenue_qoq_lag4',
    'netincome_qoq_lag1',  'netincome_qoq_lag2',  'netincome_qoq_lag4',
    'gross_margin_lag1',   'gross_margin_lag2',   'gross_margin_lag4',
    'revenue_rolling_std_4q', 'revenue_rolling_std_8q',
    'income_rolling_std_4q',  'income_rolling_std_8q',
]

# Alias dùng ở Section 8 — cùng một list, không khai báo lại
WINSORIZE_COLS = RATIO_COLS_WINSOR


# ── SignLogTransformer ────────────────────────────────────────

class SignLogTransformer(BaseEstimator, TransformerMixin):
    """
    Sign-Log Transform: f(x) = sign(x) · log(1 + |x|)

    ── Lý do học thuật ──────────────────────────────────────────────
    Dữ liệu tài chính tuyệt đối có phân phối lệch phải nặng (heavy
    right skew), vi phạm giả định chuẩn của nhiều mô hình ML.
    log() thông thường không áp dụng được khi x < 0 (công ty lỗ).

    Sign-Log giải quyết đồng thời:
      1. Giữ nguyên DẤU → không mất thông tin lãi/lỗ.
      2. log(1+|x|) → nén scale mạnh, f(0)=0, liên tục tại gốc.
      3. Hàm đơn điệu (monotone) → không thay đổi thứ tự xếp hạng.
      4. Tất định, không tham số → fit() không cần, zero leakage,
         an toàn dùng trước train/test split.

    Yeo-Johnson (method='yeo-johnson'): PowerTransformer ước lượng
    tham số λ tối ưu bằng MLE — mạnh hơn Sign-Log nhưng PHẢI fit()
    chỉ trên train để tránh leakage.

    Parameters
    ----------
    cols   : Cột cần transform. Mặc định SCALE_COLS.
    method : 'sign-log' (default, không cần fit) | 'yeo-johnson'.
    copy   : True → tạo cột mới 'log_<col>', giữ cột gốc để debug.
             False → overwrite cột gốc.
    """

    def __init__(self, cols: List[str] = None,
                 method: str = 'sign-log', copy: bool = True):
        self.cols   = cols if cols is not None else SCALE_COLS
        self.method = method
        self.copy   = copy
        self._yj_transformers: Dict[str, PowerTransformer] = {}

    def fit(self, X: pd.DataFrame, y=None) -> 'SignLogTransformer':
        if not isinstance(X, pd.DataFrame):
            raise TypeError('SignLogTransformer yêu cầu X là pd.DataFrame.')
        self._actual_cols = [c for c in self.cols if c in X.columns]

        if self.method == 'yeo-johnson':
            # Ước lượng λ tối ưu bằng MLE — CHỈ TRÊN TRAIN (anti-leakage)
            self._yj_transformers = {}
            for col in self._actual_cols:
                series = X[[col]].dropna()
                if len(series) < 10:
                    logger.warning('[SignLog.fit] "%s" < 10 mẫu → bỏ qua YJ.', col)
                    continue
                pt = PowerTransformer(method='yeo-johnson', standardize=False)
                pt.fit(series)
                self._yj_transformers[col] = pt
            logger.info('[SignLog.fit] Yeo-Johnson fit %d cột | λ (3 đầu): %s',
                        len(self._yj_transformers),
                        {c: round(self._yj_transformers[c].lambdas_[0], 3)
                         for c in list(self._yj_transformers)[:3]})
        else:
            # Sign-Log: hàm tất định, không cần fit
            logger.info('[SignLog.fit] Sign-Log tất định — %d cột (không tham số).',
                        len(self._actual_cols))
        return self

    def transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError('SignLogTransformer yêu cầu X là pd.DataFrame.')
        X = X.copy()
        applied = []
        for col in self._actual_cols:
            if col not in X.columns:
                continue
            if self.method == 'sign-log':
                # f(x) = sign(x) · log(1+|x|)
                transformed = np.sign(X[col]) * np.log1p(X[col].abs())
            else:  # yeo-johnson
                if col not in self._yj_transformers:
                    # Không có model → fallback sign-log
                    transformed = np.sign(X[col]) * np.log1p(X[col].abs())
                    logger.warning('[SignLog.transform] "%s" thiếu YJ model → fallback sign-log.', col)
                else:
                    pt         = self._yj_transformers[col]
                    mask_valid = X[col].notna()
                    transformed = X[col].copy()
                    if mask_valid.sum() > 0:
                        transformed[mask_valid] = pt.transform(
                            X.loc[mask_valid, [col]]).ravel()
            out_col = f'log_{col}' if self.copy else col
            X[out_col] = transformed
            applied.append(col)

        logger.info('[SignLog.transform] method="%s" | %d/%d cột%s',
                    self.method, len(applied), len(self._actual_cols),
                    ' → cột log_*' if self.copy else ' (overwrite)')
        return X

    def get_lambda_report(self) -> Optional[pd.DataFrame]:
        """(Chỉ Yeo-Johnson) Bảng λ tối ưu theo cột.
        λ≈0 → log | λ≈1 → tuyến tính | λ≈2 → nghịch đảo."""
        if not self._yj_transformers:
            logger.warning('[SignLog] get_lambda_report() chỉ dùng với method="yeo-johnson".')
            return None
        return (pd.DataFrame([{'column': c, 'lambda': round(pt.lambdas_[0], 4)}
                               for c, pt in self._yj_transformers.items()])
                  .set_index('column').sort_values('lambda'))


# ── handle_outliers() — chỉ thực hiện BƯỚC A (Sign-Log) ──────────
# Bước B (Sector Winsorize) được thực hiện trong sklearn Pipeline ở
# Section 8 — sau split — vì cần fit quantile chỉ trên train.
#
# Tên hàm giữ nguyên là `handle_outliers` để tương thích với
# `prepare_ml_data()` ở Section 9 (entry point gọi đầy đủ pipeline).

_sign_log_tf = SignLogTransformer(cols=SCALE_COLS, method='sign-log', copy=True)
_sign_log_tf.fit(pd.DataFrame(columns=SCALE_COLS))  # no-op cho sign-log


def handle_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Outlier handling — chỉ làm BƯỚC A (Sign-Log transform).

    Bước A (cell này):     Sign-Log transform cho biến quy mô (SCALE_COLS).
                            → Tạo cột log_<col>, giữ cột gốc.
                            → Không leakage vì Sign-Log là hàm tất định.
    Bước B (Section 8):    Sector Winsorize [1%–99%] cho biến tỷ lệ.
                            → fit quantile từ train, transform cả hai.
    """
    logger.info('[Outlier] Bước A: Sign-Log transform (%d cột quy mô)...', len(SCALE_COLS))
    df = _sign_log_tf.transform(df)
    logger.info('[Outlier] Bước A xong. Shape: %s', df.shape)
    return df


# ── Run ───────────────────────────────────────────────────────

df_fe = handle_outliers(df_fe)

logger.info('[Outlier] df_fe sau Sign-Log: %s', df_fe.shape)


20:08:50 | INFO | [SignLog.fit] Sign-Log tất định — 23 cột (không tham số).
20:08:50 | INFO | [Outlier] Bước A: Sign-Log transform (23 cột quy mô)...
20:08:50 | INFO | [SignLog.transform] method="sign-log" | 23/23 cột → cột log_*
20:08:50 | INFO | [Outlier] Bước A xong. Shape: (1761, 117)
20:08:50 | INFO | [Outlier] df_fe sau Sign-Log: (1761, 117)


---
## **6. Chia dữ liệu chuỗi thời gian (Time-Series Split)**

**Chronological split** — không random. Model chỉ học dữ liệu quá khứ để dự đoán tương lai.

- `train`: năm ≤ `TRAIN_CUTOFF` (= 2021)
- `test` : năm > `TRAIN_CUTOFF` (= 2022 trở đi)
- Tỷ lệ xấp xỉ: **~75% train / ~25% test** (tuỳ độ dài lịch sử mỗi symbol).


In [10]:
def split_time_series(
    df: pd.DataFrame,
    target_col: str = TARGET_CLS,
    cutoff_year: int = TRAIN_CUTOFF
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, List[str]]:
    """
    Chronological train/test split:
      - Loại bỏ hàng không có target hợp lệ
      - train: fiscaldateending.year <= cutoff_year
      - test : fiscaldateending.year >  cutoff_year

    Trả về (X_train, X_test, y_train, y_test, feature_cols)

    ⚠️  CLASS IMBALANCE — xem EDA notebook 03, Section 5.2b:
        Một số sector có phân phối class lệch (< 35% hoặc > 65% class "Tăng").
        Khi training model phía sau, nên xử lý imbalance bằng một trong:
          - LogisticRegression / RandomForest: class_weight='balanced'
          - XGBoost / LightGBM:                scale_pos_weight
          - Cross-validation:                  StratifiedKFold theo sector_group
        Pipeline preprocessing không tự xử lý vì việc này phụ thuộc vào model.

    ⚠️  sector_group được GIỮ LẠI trong X_train/X_test (cột string)
    để SectorWinsorizer.fit() dùng làm grouping key.
    SectorWinsorizer sẽ tự drop cột này sau khi winsorize xong,
    trước khi truyền sang SimpleImputer → StandardScaler.
    """
    df_valid = df[df[target_col].notna()].copy()
    logger.info('[Split] Valid rows (target not NaN): %d / %d',
                len(df_valid), len(df))

    if len(df_valid[df_valid[DATE_COL].dt.year > cutoff_year]) < 50:
        logger.warning('[Split] ⚠️  Test set có ít hơn 50 rows — kiểm tra lại TRAIN_CUTOFF')

    _log_cols_present = {c[4:] for c in df_valid.columns if c.startswith('log_')}
    _rolling_mean_cols = [c for c in df_valid.columns
                          if 'rolling_mean' in c]

    NON_FEATURE = set([
        GROUP_COL, DATE_COL, 'reportedcurrency', 'quarter_label',
        TARGET_REG, TARGET_CLS,
    ]) | _log_cols_present | set(_rolling_mean_cols)

    # feature_cols = numeric cols + sector_group (string, dùng cho winsorizer)
    numeric_feature_cols = [
        c for c in df_valid.columns
        if c not in NON_FEATURE
        and c != SECTOR_COL                          # tách riêng
        and df_valid[c].dtype in [np.float64, np.float32, np.int64, np.int32]
    ]
    feature_cols = [SECTOR_COL] + numeric_feature_cols  # sector đầu tiên

    X = df_valid[feature_cols].copy()
    y = df_valid[target_col].copy()

    train_mask = df_valid[DATE_COL].dt.year <= cutoff_year
    test_mask  = ~train_mask

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    logger.info('[Split] Cutoff year: %d', cutoff_year)
    logger.info('[Split] Train: %d rows (%d numeric features + sector_group) | %s → %s',
                len(X_train), len(numeric_feature_cols),
                df_valid[train_mask][DATE_COL].min().date(),
                df_valid[train_mask][DATE_COL].max().date())
    logger.info('[Split] Test : %d rows | %s → %s',
                len(X_test),
                df_valid[test_mask][DATE_COL].min().date(),
                df_valid[test_mask][DATE_COL].max().date())
    logger.info('[Split] Train/Test ratio: %.1f%% / %.1f%%',
                len(X_train)/(len(X_train)+len(X_test))*100,
                len(X_test) /(len(X_train)+len(X_test))*100)

    # Cảnh báo class balance — gợi ý dùng class_weight ở bước training nếu lệch
    if target_col == TARGET_CLS:
        train_dist = y_train.value_counts(normalize=True).sort_index()
        if len(train_dist) >= 2:
            minor_pct = train_dist.min() * 100
            if minor_pct < 35:
                logger.warning('[Split] ⚠️  Train class imbalance: class thiểu số = %.1f%% '
                               '→ nên dùng class_weight="balanced" khi training', minor_pct)
            else:
                logger.info('[Split] Train class balance OK: %.1f%% / %.1f%%',
                            train_dist.iloc[0]*100, train_dist.iloc[1]*100)

    return X_train, X_test, y_train, y_test, numeric_feature_cols


X_train, X_test, y_train, y_test, FEATURE_COLS = split_time_series(df_fe)

print(f'\nNumeric feature list ({len(FEATURE_COLS)} features):')
for i, f in enumerate(FEATURE_COLS, 1):
    print(f'  {i:3d}. {f}')
print(f'  (+ sector_group column for SectorWinsorizer grouping)')


20:08:50 | INFO | [Split] Valid rows (target not NaN): 1735 / 1761
20:08:50 | INFO | [Split] Cutoff year: 2021
20:08:50 | INFO | [Split] Train: 1368 rows (77 numeric features + sector_group) | 2006-01-31 → 2021-12-31
20:08:50 | INFO | [Split] Test : 367 rows | 2022-01-31 → 2025-12-31
20:08:50 | INFO | [Split] Train/Test ratio: 78.8% / 21.2%
20:08:50 | INFO | [Split] Train class balance OK: 45.5% / 54.5%



Numeric feature list (77 features):
    1. interestexpense
    2. netincomefromcontinuingoperations
    3. is_usd
    4. year
    5. quarter
    6. gross_margin
    7. operating_margin
    8. net_margin
    9. ebitda_margin
   10. revenue_qoq
   11. netincome_qoq
   12. grossprofit_qoq
   13. revenue_yoy_growth
   14. income_yoy_growth
   15. grossprofit_yoy_growth
   16. revenue_rolling_std_4q
   17. revenue_rolling_std_8q
   18. revenue_rolling_max_4q
   19. revenue_rolling_min_4q
   20. income_rolling_std_4q
   21. income_rolling_std_8q
   22. income_rolling_max_4q
   23. income_rolling_min_4q
   24. net_margin_rolling_std_4q
   25. net_margin_rolling_std_8q
   26. net_margin_rolling_max_4q
   27. net_margin_rolling_min_4q
   28. gross_margin_rolling_std_4q
   29. gross_margin_rolling_std_8q
   30. gross_margin_rolling_max_4q
   31. gross_margin_rolling_min_4q
   32. revenue_qoq_rolling_std_4q
   33. revenue_qoq_rolling_std_8q
   34. revenue_qoq_rolling_max_4q
   35. revenue_qoq_ro

---
## **7. Imputation & Winsorization**

In [11]:
# Cột nhóm 1: điền 0
ZERO_FILL_COLS = [
    'researchanddevelopment', 'interestexpense', 'interestincome',
    'netinterestincome', 'noninterestincome', 'othernonoperatingincome',
    'investmentincomenet', 'interestanddebtexpense',
    'sellinggeneralandadministrative', 'depreciation',
    'depreciationandamortization',
]

# Cột nhóm 2: ffill/bfill + sector median
FFILL_COLS = [
    'totalrevenue', 'grossprofit', 'operatingincome', 'netincome',
    'costofrevenue', 'costofgoodsandservicessold', 'operatingexpenses',
    'incomebeforetax', 'incometaxexpense', 'ebit', 'ebitda',
    'netincomefromcontinuingoperations', 'comprehensiveincomenetoftax',
    'gross_margin', 'operating_margin', 'net_margin', 'ebitda_margin',
    'revenue_qoq', 'netincome_qoq', 'grossprofit_qoq',
    'revenue_yoy_growth', 'income_yoy_growth', 'grossprofit_yoy_growth',
    'revenue_momentum', 'earnings_momentum',
    'revenue_growth_volatility_4q', 'income_growth_volatility_4q',
    'log_totalrevenue', 'log_grossprofit', 'log_operatingincome',
    'log_netincome', 'log_ebit', 'log_ebitda',
]


class Imputer:
    """
    Điền khuyết nâng cao — chống Data Leakage.
    Pattern: fit(train) → transform(train) → transform(test).
    """

    def __init__(self, symbol_col: str = 'symbol',
                 sector_col:  str = SECTOR_COL,
                 zero_cols:   List[str] = None,
                 ffill_cols:  List[str] = None):
        self.symbol_col = symbol_col
        self.sector_col = sector_col
        self.zero_cols  = zero_cols  if zero_cols  is not None else ZERO_FILL_COLS
        self.ffill_cols = ffill_cols if ffill_cols is not None else FFILL_COLS
        self._sector_medians: Dict[str, Dict[str, float]] = {}
        self._global_medians: Dict[str, float]            = {}
        self._fitted = False

    def fit(self, df_train: pd.DataFrame) -> 'Imputer':
        """Học sector median CHỈ TRÊN df_train."""
        actual_ffill = [c for c in self.ffill_cols if c in df_train.columns]

        # Global fallback: median toàn train
        for col in actual_ffill:
            self._global_medians[col] = float(df_train[col].median())

        # Per-sector median (chỉ từ train)
        if self.sector_col in df_train.columns:
            for sector, grp in df_train.groupby(self.sector_col):
                self._sector_medians[sector] = {}
                for col in actual_ffill:
                    med = grp[col].median()
                    self._sector_medians[sector][col] = (
                        float(med) if not np.isnan(med)
                        else self._global_medians.get(col, 0.0)
                    )
        else:
            logger.warning('[Imputer.fit] Cột ngành "%s" không có → dùng global median.',
                           self.sector_col)

        self._fitted = True
        logger.info('[Imputer.fit] %d sectors | %d ffill-cols | %d zero-cols',
                    len(self._sector_medians),
                    len(actual_ffill),
                    len([c for c in self.zero_cols if c in df_train.columns]))
        return self

    def transform(self, df: pd.DataFrame, split_label: str = 'set') -> pd.DataFrame:
        """Áp dụng imputation. Không tính thống kê mới từ df → zero leakage."""
        if not self._fitted:
            raise RuntimeError('Imputer chưa fit. Gọi .fit(df_train) trước.')
        df = df.copy()

        # ── Nhóm 1: Zero-fill ─────────────────────────────────────
        n_zero = 0
        for col in self.zero_cols:
            if col in df.columns:
                n = df[col].isna().sum()
                if n > 0:
                    df[col] = df[col].fillna(0)
                    n_zero += n
        logger.info('[Imputer.%s] Nhóm 1 Zero-fill: %d NaN → 0', split_label, n_zero)

        # ── Nhóm 2a: ffill → bfill theo symbol ────────────────────
        # Kế thừa lịch sử của chính công ty đó (không mix sang công ty khác).
        actual_ffill = [c for c in self.ffill_cols if c in df.columns]
        nan_b = df[actual_ffill].isna().sum().sum()
        if self.symbol_col in df.columns:
            df[actual_ffill] = (
                df.groupby(self.symbol_col)[actual_ffill]
                  .transform(lambda g: g.ffill().bfill())
            )
        nan_a = df[actual_ffill].isna().sum().sum()
        logger.info('[Imputer.%s] Nhóm 2a ffill→bfill: %d → %d NaN (điền %d)',
                    split_label, nan_b, nan_a, nan_b - nan_a)

        # ── Nhóm 2b: Sector median (từ train) ─────────────────────
        # VECTORIZED FIX: dùng .map(dict) thay vì .apply(axis=1) → nhanh ~100x
        # khi dataset lớn.
        n_sec = 0
        has_sector = self.sector_col in df.columns
        for col in actual_ffill:
            missing_mask = df[col].isna()
            n_missing = int(missing_mask.sum())
            if n_missing == 0:
                continue

            if has_sector:
                # Tạo dict {sector: median_col} — fallback global nếu thiếu
                global_med  = self._global_medians.get(col, 0.0)
                sector_map  = {
                    sec: meds.get(col, global_med)
                    for sec, meds in self._sector_medians.items()
                }
                # Vectorized: map sector → median, NaN sector → global
                df.loc[missing_mask, col] = (
                    df.loc[missing_mask, self.sector_col]
                      .map(sector_map)
                      .fillna(global_med)
                )
            else:
                df.loc[missing_mask, col] = self._global_medians.get(col, 0.0)
            n_sec += n_missing
        logger.info('[Imputer.%s] Nhóm 2b sector median (train): %d NaN đã điền',
                    split_label, n_sec)

        # ── Cảnh báo NaN còn sót ──────────────────────────────────
        remaining = df[actual_ffill].isna().sum()
        remaining = remaining[remaining > 0]
        if len(remaining):
            logger.warning('[Imputer.%s] Còn NaN:\n%s', split_label, remaining.to_string())
        else:
            logger.info('[Imputer.%s] Tất cả NaN đã được điền.', split_label)
        return df

    def fit_transform(self, df_train: pd.DataFrame) -> pd.DataFrame:
        """Shortcut: fit trên train rồi transform train."""
        return self.fit(df_train).transform(df_train, split_label='train')

    def get_sector_medians(self) -> pd.DataFrame:
        """Bảng sector median đã học — dùng để kiểm tra/debug."""
        if not self._sector_medians:
            return pd.DataFrame()
        return pd.DataFrame(self._sector_medians).T


# ── sklearn-compatible wrapper cho Imputer ────────────────────
class ImputerStep(BaseEstimator, TransformerMixin):
    """
    sklearn-compatible wrapper cho Imputer.
    Pattern: fit(X_train) → transform(X_train) → transform(X_test).

    Quy ước label log:
      - fit_transform()    → label='train' (1 lần, rõ ràng)
      - transform() lần 1  sau fit  → label='train' (gọi từ fit_transform fallback)
      - transform() lần 2+ sau fit  → label='test'

    Counter được RESET trong fit() để mỗi fit mới bắt đầu lại đếm,
    tránh tình trạng "test" bị label nhầm thành "train" khi re-fit.
    """
    def __init__(self, symbol_col: str = 'symbol',
                 sector_col: str = SECTOR_COL,
                 zero_cols: List[str] = None,
                 ffill_cols: List[str] = None):
        self.symbol_col = symbol_col
        self.sector_col = sector_col
        self.zero_cols  = zero_cols
        self.ffill_cols = ffill_cols
        self._imputer: Imputer = None
        self._n_transforms: int = 0

    def fit(self, X: pd.DataFrame, y=None) -> 'ImputerStep':
        self._imputer = Imputer(
            symbol_col=self.symbol_col,
            sector_col=self.sector_col,
            zero_cols=self.zero_cols,
            ffill_cols=self.ffill_cols,
        )
        self._imputer.fit(X)
        self._n_transforms = 0   # FIX: reset counter ở mỗi fit()
        return self

    def transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        # Counter rõ ràng: lần đầu sau fit = train, các lần sau = test.
        label = 'train' if self._n_transforms == 0 else 'test'
        self._n_transforms += 1
        return self._imputer.transform(X, split_label=label)

    def fit_transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        self.fit(X)
        # Gọi trực tiếp _imputer.transform để giữ label='train' rõ ràng
        # mà KHÔNG tăng _n_transforms (nếu tăng thì lần transform() kế tiếp
        # sẽ bị nhãn 'test' chính xác).
        self._n_transforms = 1
        return self._imputer.transform(X, split_label='train')

    def get_sector_medians(self) -> pd.DataFrame:
        return self._imputer.get_sector_medians() if self._imputer else pd.DataFrame()


# ── Tái tạo df_train_raw / df_test_raw từ df_fe ──────────────
_train_mask  = df_fe[DATE_COL].dt.year <= TRAIN_CUTOFF
_valid_mask  = df_fe[TARGET_CLS].notna()
df_train_raw = df_fe[_train_mask & _valid_mask].copy()
df_test_raw  = df_fe[~_train_mask & _valid_mask].copy()

logger.info('[Section 7] df_train_raw: %s | df_test_raw: %s',
            df_train_raw.shape, df_test_raw.shape)

# ── Imputation với ImputerStep ────────────────────────────────
adv_imputer      = ImputerStep()
df_train_imputed = adv_imputer.fit_transform(df_train_raw)
df_test_imputed  = adv_imputer.transform(df_test_raw)

print('\nSector medians học từ train (5 cột đầu, 5 ngành đầu):')
display(adv_imputer.get_sector_medians().iloc[:5, :5].round(4))

# ── Cập nhật X_train / X_test ────────────────────────────────
_feature_cols_with_ind = [SECTOR_COL] + [
    c for c in X_train.columns if c != SECTOR_COL
]
X_train = df_train_imputed[[c for c in _feature_cols_with_ind if c in df_train_imputed.columns]]
X_test  = df_test_imputed[[c for c in _feature_cols_with_ind if c in df_test_imputed.columns]]

logger.info('[Section 7] X_train sau impute (chưa winsorize): %s | X_test: %s',
            X_train.shape, X_test.shape)


20:08:50 | INFO | [Section 7] df_train_raw: (1368, 117) | df_test_raw: (367, 117)
20:08:50 | INFO | [Imputer.fit] 7 sectors | 32 ffill-cols | 4 zero-cols
20:08:50 | INFO | [Imputer.train] Nhóm 1 Zero-fill: 289 NaN → 0
20:08:50 | INFO | [Imputer.train] Nhóm 2a ffill→bfill: 818 → 0 NaN (điền 818)
20:08:50 | INFO | [Imputer.train] Nhóm 2b sector median (train): 0 NaN đã điền
20:08:50 | INFO | [Imputer.train] Tất cả NaN đã được điền.
20:08:50 | INFO | [Imputer.test] Nhóm 1 Zero-fill: 139 NaN → 0
20:08:50 | INFO | [Imputer.test] Nhóm 2a ffill→bfill: 2 → 0 NaN (điền 2)
20:08:50 | INFO | [Imputer.test] Nhóm 2b sector median (train): 0 NaN đã điền
20:08:50 | INFO | [Imputer.test] Tất cả NaN đã được điền.



Sector medians học từ train (5 cột đầu, 5 ngành đầu):


,totalrevenue,grossprofit,operatingincome,netincome,costofrevenue
Communication Services,5374600000.0000,2879931000.0000,1373964000.0000,851000000.0000,2047490000.0000
Consumer Discretionary,14588000000.0000,4874380000.0000,883276500.0000,629000000.0000,10156000000.0000
Consumer Staples,71175500000.0000,13690500000.0000,3370000000.0000,1356500000.0000,57485000000.0000
Energy,83658500000.0000,21006000000.0000,7908500000.0000,6805000000.0000,55436500000.0000
Financials,22763000000.0000,4023000000.0000,3142000000.0000,2373000000.0000,1064500000.0000


20:08:50 | INFO | [Section 7] X_train sau impute (chưa winsorize): (1368, 78) | X_test: (367, 78)


---
## **8. sklearn Preprocessing Pipeline**

Bước này chạy `SectorWinsorizer → StandardScaler` trên feature matrix. Cả hai đều `fit()` chỉ trên train.

**Vì sao winsorize theo `sector_group` (7 ngành) thay vì theo `symbol` (25 công ty)?**

EDA notebook 03 — Section 4.6 (Panel Data) cho thấy scale tài chính khác nhau lớn giữa các symbol → cần group-wise normalization thay vì global. Nhưng nếu winsorize theo `symbol`:
- Mỗi symbol chỉ có 30–60 quan sát → quá ít để tính quantile [1%, 99%] đáng tin cậy
- Một bất thường thật sự trong lịch sử công ty (ví dụ quý lỗ kỷ lục) sẽ bị clip sai

Winsorize theo `sector_group` cân bằng được 2 yêu cầu:
- Đủ mẫu mỗi nhóm (~200–500 quan sát/sector) → quantile ổn định
- Vẫn tôn trọng đặc thù ngành (Financials margin khác Tech khác Energy)

`min_samples=30` được dùng làm ngưỡng fallback — nếu sector có < 30 mẫu thì dùng quantile toàn thị trường (`global_bounds_`).


In [12]:
# ── SectorWinsorizer ──────────────────────────────────────────
# Winsorize biến tỷ lệ theo từng nhóm ngành.
# WINSORIZE_COLS đã định nghĩa ở Section 5 (alias của RATIO_COLS_WINSOR).

class SectorWinsorizer(BaseEstimator, TransformerMixin):
    """
    Winsorize biến tỷ lệ [lower, upper] theo từng nhóm ngành.
    fit() chỉ trên train → quantile từ train, không rò rỉ test.
    transform() trả về np.ndarray (đã drop cột sector_col).
    """
    def __init__(self, cols: List[str], lower: float = 0.01, upper: float = 0.99,
                 sector_col: str = SECTOR_COL, min_samples: int = 30):
        self.cols = cols
        self.lower = lower
        self.upper = upper
        self.sector_col = sector_col
        self.min_samples = min_samples
        self.bounds_ = {}
        self.global_bounds_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("SectorWinsorizer yêu cầu X là pd.DataFrame")

        self.actual_cols_ = [c for c in self.cols if c in X.columns]

        # Ngưỡng fallback toàn thị trường (Global) tính từ tập Train
        for col in self.actual_cols_:
            self.global_bounds_[col] = (
                float(X[col].quantile(self.lower)),
                float(X[col].quantile(self.upper))
            )

        # Ngưỡng phân tầng chi tiết theo từng nhóm ngành học trên tập Train
        for sector, grp in X.groupby(self.sector_col):
            self.bounds_[sector] = {}
            for col in self.actual_cols_:
                col_data = grp[col].dropna()
                if len(col_data) >= self.min_samples:
                    lo = float(col_data.quantile(self.lower))
                    hi = float(col_data.quantile(self.upper))
                else:
                    lo, hi = self.global_bounds_[col]
                self.bounds_[sector][col] = (lo, hi)

        logger.info('[SectorWinsorizer 1%%-99%%] fit() hoàn tất: %d ngành, %d cột',
                    len(self.bounds_), len(self.actual_cols_))
        return self

    def transform(self, X: pd.DataFrame, y=None) -> np.ndarray:
        X_out = X.copy()
        unseen = set()

        for sector, grp_idx in X_out.groupby(self.sector_col).groups.items():
            if sector not in self.bounds_:
                unseen.add(sector)
                bounds = self.global_bounds_
            else:
                bounds = self.bounds_[sector]

            for col in self.actual_cols_:
                if col in X_out.columns:
                    lo, hi = bounds.get(col, self.global_bounds_[col])
                    X_out.loc[grp_idx, col] = X_out.loc[grp_idx, col].clip(lower=lo, upper=hi)

        if unseen:
            logger.warning('[SectorWinsorizer] Gặp %d ngành lạ ở tập test -> Dùng global fallback: %s',
                           len(unseen), list(unseen))

        # DROP cột sector_col tại đây để đầu ra chỉ còn mảng số tương thích với Scaler
        X_numeric = X_out.drop(columns=[self.sector_col], errors='ignore')
        return X_numeric.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.array([f for f in input_features if f != self.sector_col])
        return np.array(self.actual_cols_)


# ── Build preprocessing pipeline ──────────────────────────────
# Sign-Log đã làm ở Section 5 (trước split, deterministic).
# Ở đây chỉ cần: Winsorize → Scale.

def build_preprocessing_pipeline(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    winsorize_cols: List[str] = None,
) -> Tuple[np.ndarray, np.ndarray, List[str], Pipeline]:
    """
    Xây dựng và fit preprocessing pipeline.
    Thứ tự: SectorWinsorizer (ratio cols) → StandardScaler.

    fit() chỉ trên X_train, transform() trên cả train và test.
    """
    if winsorize_cols is None:
        winsorize_cols = WINSORIZE_COLS

    numeric_pipeline = Pipeline(steps=[
        ('winsorizer', SectorWinsorizer(cols=winsorize_cols, sector_col=SECTOR_COL,
                                         lower=0.01, upper=0.99)),  # Winsorize [1%–99%] & drop sector col
        ('scaler',     StandardScaler()),                           # Chuẩn hóa Z-score
    ])

    # FIT CHỈ TRÊN TRAIN, TRANSFORM TRÊN CẢ HAI
    X_train_scaled = numeric_pipeline.fit_transform(X_train)
    X_test_scaled  = numeric_pipeline.transform(X_test)

    # Tên features đầu ra (loại sector_col)
    feature_names = [c for c in X_train.columns if c != SECTOR_COL]

    logger.info('[Pipeline] Các bước: %s', [s[0] for s in numeric_pipeline.steps])
    logger.info('[Pipeline] X_train_scaled: %s', X_train_scaled.shape)
    logger.info('[Pipeline] X_test_scaled : %s', X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, feature_names, numeric_pipeline


# Chạy pipeline tổng hợp
X_train_scaled, X_test_scaled, feature_names, pipeline = build_preprocessing_pipeline(X_train, X_test)

print(f'\nPipeline built successfully!')
print(f'   X_train_scaled: {X_train_scaled.shape}')
print(f'   X_test_scaled : {X_test_scaled.shape}')
print('\n   Các bước xử lý:', [s[0] for s in pipeline.steps])


20:08:50 | INFO | [SectorWinsorizer 1%-99%] fit() hoàn tất: 7 ngành, 30 cột
20:08:50 | INFO | [Pipeline] Các bước: ['winsorizer', 'scaler']
20:08:50 | INFO | [Pipeline] X_train_scaled: (1368, 77)
20:08:50 | INFO | [Pipeline] X_test_scaled : (367, 77)



Pipeline built successfully!
   X_train_scaled: (1368, 77)
   X_test_scaled : (367, 77)

   Các bước xử lý: ['winsorizer', 'scaler']


---
## **9. Feature Selection**

In [13]:
def remove_correlated_features(
    X_train_scaled: np.ndarray,
    X_test_scaled: np.ndarray,
    feature_names: List[str],
    threshold: float = 0.95
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    """
    Loại bỏ feature có correlation với feature khác > threshold.
    Chỉ tính correlation trên TRAIN set để tránh leakage.

    Thuật toán:
      - Tính correlation matrix trên train
      - Với mỗi cặp (i, j) có |corr| > threshold → giữ i, loại j
      - Loại thêm feature có variance ≈ 0

    Returns:
        X_train_sel, X_test_sel, selected_features, dropped_features
    """
    # ── Bước 1: Correlation threshold ────────────────────────
    X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)
    corr_matrix = X_train_df.corr().abs()

    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
    )

    to_drop_corr = [
        col for col in upper.columns
        if any(upper[col] > threshold)
    ]
    logger.info('[FeatureSel] Correlation > %.2f → dropping %d features: %s',
                threshold, len(to_drop_corr), to_drop_corr[:10])

    # ── Bước 2: Low variance filter ──────────────────────────
    vt = VarianceThreshold(threshold=0.001)
    vt.fit(X_train_df.drop(columns=to_drop_corr, errors='ignore'))
    low_var_mask = vt.get_support()
    remaining_after_corr = [f for f in feature_names if f not in to_drop_corr]
    to_drop_var = [
        f for f, keep in zip(remaining_after_corr, low_var_mask) if not keep
    ]
    logger.info('[FeatureSel] Low variance → dropping %d features: %s',
                len(to_drop_var), to_drop_var)

    # ── Combine drops ─────────────────────────────────────────
    all_drop = set(to_drop_corr) | set(to_drop_var)
    selected = [f for f in feature_names if f not in all_drop]

    keep_idx = [feature_names.index(f) for f in selected]
    X_train_sel = X_train_scaled[:, keep_idx]
    X_test_sel  = X_test_scaled[:, keep_idx]

    logger.info('[FeatureSel] Selected %d / %d features',
                len(selected), len(feature_names))

    return X_train_sel, X_test_sel, selected, list(all_drop)


X_train_sel, X_test_sel, SELECTED_FEATURES, DROPPED_FEATURES = \
    remove_correlated_features(X_train_scaled, X_test_scaled, feature_names)

print(f'\nFeature selection done!')
print(f'   Before : {len(feature_names)} features')
print(f'   Dropped: {len(DROPPED_FEATURES)} features')
print(f'   Final  : {len(SELECTED_FEATURES)} features')
print(f'\nFinal feature list:')
for i, f in enumerate(SELECTED_FEATURES, 1):
    print(f'  {i:3d}. {f}')


20:08:50 | INFO | [FeatureSel] Correlation > 0.95 → dropping 10 features: ['net_margin', 'revenue_rolling_min_4q', 'net_margin_rolling_min_4q', 'revenue_qoq_rolling_max_4q', 'operating_margin_lag1', 'operating_margin_lag4', 'log_costofgoodsandservicessold', 'log_totalrevenue_lag1', 'log_totalrevenue_lag2', 'log_totalrevenue_lag4']
20:08:50 | INFO | [FeatureSel] Low variance → dropping 0 features: []
20:08:50 | INFO | [FeatureSel] Selected 67 / 77 features



Feature selection done!
   Before : 77 features
   Dropped: 10 features
   Final  : 67 features

Final feature list:
    1. interestexpense
    2. netincomefromcontinuingoperations
    3. is_usd
    4. year
    5. quarter
    6. gross_margin
    7. operating_margin
    8. ebitda_margin
    9. revenue_qoq
   10. netincome_qoq
   11. grossprofit_qoq
   12. revenue_yoy_growth
   13. income_yoy_growth
   14. grossprofit_yoy_growth
   15. revenue_rolling_std_4q
   16. revenue_rolling_std_8q
   17. revenue_rolling_max_4q
   18. income_rolling_std_4q
   19. income_rolling_std_8q
   20. income_rolling_max_4q
   21. income_rolling_min_4q
   22. net_margin_rolling_std_4q
   23. net_margin_rolling_std_8q
   24. net_margin_rolling_max_4q
   25. gross_margin_rolling_std_4q
   26. gross_margin_rolling_std_8q
   27. gross_margin_rolling_max_4q
   28. gross_margin_rolling_min_4q
   29. revenue_qoq_rolling_std_4q
   30. revenue_qoq_rolling_std_8q
   31. revenue_qoq_rolling_min_4q
   32. revenue_moment

In [14]:
# prepare_ml_data()

def prepare_ml_data(
    filepath: str = FILE_PATH,
    target_col: str = TARGET_CLS,
    cutoff_year: int = TRAIN_CUTOFF,
    corr_threshold: float = 0.95
) -> dict:
    """
    Entry point toàn bộ pipeline (đầy đủ, gọi độc lập được):
      1. load_data()
      2. clean_data()         ← Chuẩn hóa, duplicate, tiền tệ
      3. engineer_features()  ← Margin, QoQ, YoY, rolling, lag
      4. create_target()      ← next_quarter_net_income, profit_increase_next
      5. handle_outliers()    ← Sign-Log transform (tất định, trước split)
      6. split_time_series()  ← Chronological train/test split
      7. ImputerStep          ← fit trên train → transform cả hai  [FIX]
      8. build_preprocessing_pipeline()  ← Winsorize [1%–99%] + StandardScaler
      9. remove_correlated_features()    ← Correlation & low-variance filter

    Mọi fit() đều CHỈ trên X_train — không có data leakage.

    Returns dict:
      {'X_train', 'X_test', 'y_train', 'y_test',
       'feature_names', 'pipeline', 'imputer', 'df_processed'}
    """
    logger.info('=' * 55)
    logger.info('Starting full ML data preparation pipeline')
    logger.info('=' * 55)

    df = load_data(filepath)
    df = clean_data(df)
    df = engineer_features(df)
    df = create_target(df)
    df = handle_outliers(df)

    # ── Chronological split ──────────────────────────
    X_tr, X_te, y_tr, y_te, feat_cols = split_time_series(df, target_col, cutoff_year)

    # ── Imputation (FIX — bước này trước đây bị thiếu) ──
    _imputer = ImputerStep()
    # Impute trên df đầy đủ theo split mask để giữ symbol/sector context
    _train_mask = df[DATE_COL].dt.year <= cutoff_year
    _valid_mask  = df[target_col].notna()
    df_tr_raw = df[_train_mask & _valid_mask].copy()
    df_te_raw = df[~_train_mask & _valid_mask].copy()

    df_tr_imp = _imputer.fit_transform(df_tr_raw)
    df_te_imp = _imputer.transform(df_te_raw)

    # FIX: lưu CSV SAU khi impute — file chứa dữ liệu đã sạch hoàn toàn
    df_full_imputed = pd.concat([df_tr_imp, df_te_imp]).sort_values([GROUP_COL, DATE_COL])
    df_full_imputed.to_csv(PROCESSED_PATH, index=False, encoding='utf-8-sig')
    logger.info('Saved processed_data.csv (sau impute) → %s', df_full_imputed.shape)

    # Cập nhật X_tr / X_te từ df đã impute
    _feat_with_sector = [SECTOR_COL] + [c for c in X_tr.columns if c != SECTOR_COL]
    X_tr = df_tr_imp[[c for c in _feat_with_sector if c in df_tr_imp.columns]]
    X_te = df_te_imp[[c for c in _feat_with_sector if c in df_te_imp.columns]]
    logger.info('[prepare_ml_data] Sau impute — X_tr: %s | X_te: %s', X_tr.shape, X_te.shape)

    # ── Winsorize + Scale ─────────────────────────────
    X_tr_s, X_te_s, feat_names, pipe = build_preprocessing_pipeline(X_tr, X_te)

    # ── Feature selection ─────────────────────────────
    X_tr_f, X_te_f, sel_feats, dropped_feats = remove_correlated_features(
        X_tr_s, X_te_s, feat_names, corr_threshold
    )
    

    # DataFrame train sau feature selection
    df_train_fs = pd.DataFrame(X_tr_f, columns=sel_feats)

    logger.info('=' * 55)
    logger.info('Pipeline complete.')
    logger.info('  X_train: %s | X_test: %s', X_tr_f.shape, X_te_f.shape)
    logger.info('  y_train class dist: %s',
                dict(pd.Series(y_tr.values).value_counts()))
    logger.info('  Dropped features: %s', dropped_feats)
    logger.info('=' * 55)

    return {
        'X_train'      : X_tr_f,
        'X_test'       : X_te_f,
        'y_train'      : y_tr,
        'y_test'       : y_te,
        'feature_names': sel_feats,
        'pipeline'     : pipe,
        'imputer'      : _imputer,   # expose để debug / re-use
        'df_processed' : df,
        'df_train_fs'  : df_train_fs,
        'dropped_features': dropped_feats
    }


# ── Quick ML readiness check ──────────────────────────────────
print('Dữ liệu sẵn sàng cho ML:')
print(f'   X_train : {X_train_sel.shape}')
print(f'   X_test  : {X_test_sel.shape}')
print(f'   y_train : {y_train.shape} | classes: {sorted(y_train.unique())}')
print(f'   y_test  : {y_test.shape}')
print()
print('Plug vào model ngay:')
print('   from sklearn.ensemble import RandomForestClassifier')
print('   clf = RandomForestClassifier(random_state=42)')
print('   clf.fit(X_train_sel, y_train)')
print('   clf.score(X_test_sel, y_test)')


Dữ liệu sẵn sàng cho ML:
   X_train : (1368, 67)
   X_test  : (367, 67)
   y_train : (1368,) | classes: [np.float64(0.0), np.float64(1.0)]
   y_test  : (367,)

Plug vào model ngay:
   from sklearn.ensemble import RandomForestClassifier
   clf = RandomForestClassifier(random_state=42)
   clf.fit(X_train_sel, y_train)
   clf.score(X_test_sel, y_test)


In [15]:
# Chạy toàn bộ pipeline xử lý dữ liệu
data_dict = prepare_ml_data()

# Giải nén kết quả
X_train_sel = data_dict['X_train']
X_test_sel  = data_dict['X_test']
y_train     = data_dict['y_train']
y_test      = data_dict['y_test']
# Dữ liệu sau Feature Selection
df_train_fs = data_dict['df_train_fs']
df_train_fs.to_csv(TRAIN_FS_PATH, index=False, encoding='utf-8-sig')

# Thông tin feature selection
selected_features = data_dict['feature_names']
dropped_features  = data_dict['dropped_features']
# ── Pipeline output summary ───────────────────────────────────────────────────
print('Files saved:')
for f in [PROCESSED_PATH, TRAIN_FS_PATH]:
    f_path = Path(f).resolve()
    exists = '✅' if f_path.exists() else '❌'
    print(f'  {exists} {f}')

print('\n' + '='*50)
print('✅ Dữ liệu sẵn sàng cho ML:')
print(f'   X_train : {X_train_sel.shape}')
print(f'   X_test  : {X_test_sel.shape}')
print(f'   y_train : {y_train.shape} | classes: {sorted(y_train.unique())}')
print(f'   y_test  : {y_test.shape}')


20:08:50 | INFO | =======================================================
20:08:50 | INFO | Starting full ML data preparation pipeline
20:08:50 | INFO | =======================================================
20:08:50 | INFO | Đã loại bỏ 1 dòng rác bị khuyết reportedCurrency.
20:08:50 | INFO | Loaded "d:\DS108\ds108\data\raw\Data_DS108_raw.csv" → shape sau khi lọc rác (1761, 27)
20:08:50 | INFO | [Clean] Columns normalized: ['symbol', 'fiscaldateending', 'reportedcurrency', 'totalrevenue', 'costofrevenue', 'costofgoodsandservicessold', 'grossprofit', 'sellinggeneralandadministrative', 'researchanddevelopment', 'operatingexpenses', 'operatingincome', 'investmentincomenet', 'netinterestincome', 'interestincome', 'interestexpense', 'noninterestincome', 'othernonoperatingincome', 'depreciation', 'depreciationandamortization', 'incomebeforetax', 'incometaxexpense', 'interestanddebtexpense', 'netincomefromcontinuingoperations', 'comprehensiveincomenetoftax', 'ebit', 'ebitda', 'netincome']
20

Files saved:
  ✅ d:\DS108\ds108\data\processed\Data_DS108_processed.csv
  ✅ d:\DS108\ds108\data\processed\train_after_feature_selection.csv

✅ Dữ liệu sẵn sàng cho ML:
   X_train : (1368, 67)
   X_test  : (367, 67)
   y_train : (1368,) | classes: [np.float64(0.0), np.float64(1.0)]
   y_test  : (367,)


---
## **10. Pipeline Summary**

Kiểm tra nhanh kết quả đầu ra sau toàn bộ pipeline — shape, số feature,
tỷ lệ drop, và file processed đã được lưu chưa.


In [16]:
print('=' * 65)
print('✅ PIPELINE SUMMARY')
print('=' * 65)

steps = [
    ('Raw data',            df_raw.shape),
    ('After clean',         df_clean.shape),
    ('After feature eng',   df_fe.shape),
    ('X_train (scaled)',    X_train_scaled.shape),
    ('X_test  (scaled)',    X_test_scaled.shape),
    ('X_train (selected)',  X_train_sel.shape),
    ('X_test  (selected)',  X_test_sel.shape),
]
for name, shape in steps:
    print(f'  {name:<28} → {shape}')

print(f'\n  Total features (before selection): {len(feature_names)}')
print(f'  Dropped (corr/var)              : {len(DROPPED_FEATURES)}')
print(f'  Final selected features          : {len(SELECTED_FEATURES)}')

print(f'\n  Train period: {df_fe[df_fe["year"] <= TRAIN_CUTOFF][DATE_COL].min().date()} '
      f'→ {df_fe[df_fe["year"] <= TRAIN_CUTOFF][DATE_COL].max().date()}')
print(f'  Test  period: {df_fe[df_fe["year"] > TRAIN_CUTOFF][DATE_COL].min().date()} '
      f'→ {df_fe[df_fe["year"] > TRAIN_CUTOFF][DATE_COL].max().date()}')

print('\n📁 Files saved:')
for f in [PROCESSED_PATH]:
    f_path = Path(f).resolve()
    exists = '✅' if f_path.exists() else '❌'
    print(f'  {exists} {f}')

print('\n🔑 Data Leakage Prevention Summary:')
leakage_notes = [
    'shift(-1) dùng để tạo target, KHÔNG dùng trong features',
    'Mọi shift(n)/rolling dùng groupby(symbol) — không mix company',
    'rolling() dùng shift(1) trước → không dùng giá trị hiện tại',
    '[Section 2] resolve_business_duplicates() giữ bản ít NaN nhất, mới nhất',
    '[Section 5] SignLogTransformer tất định → an toàn trước split, zero leakage',
    '[Section 7] ImputerStep.fit() chỉ trên train → sector median từ train only',
    '[Section 7] ffill/bfill theo groupby(symbol) → không mix dữ liệu công ty khác',
    '[Section 8] SectorWinsorizer.fit() chỉ trên X_train → quantile [1%,99%] từ train',
    '[Section 8] StandardScaler.fit() chỉ trên X_train',
    '[Section 9] Correlation threshold tính trên train set only',
    'safe_divide() dùng .mask() → không raise RuntimeWarning',
    'WINSORIZE_COLS = alias của RATIO_COLS_WINSOR — nhất quán toàn notebook',
    'NON_FEATURE tự động detect log_* và rolling_mean cols — không hardcode',
    'prepare_ml_data() tích hợp đầy đủ ImputerStep — gọi độc lập an toàn',
]
for note in leakage_notes:
    print(f'  ✓ {note}')


✅ PIPELINE SUMMARY
  Raw data                     → (1761, 27)
  After clean                  → (1761, 21)
  After feature eng            → (1761, 117)
  X_train (scaled)             → (1368, 77)
  X_test  (scaled)             → (367, 77)
  X_train (selected)           → (1368, 67)
  X_test  (selected)           → (367, 67)

  Total features (before selection): 77
  Dropped (corr/var)              : 10
  Final selected features          : 67

  Train period: 2006-01-31 → 2021-12-31
  Test  period: 2022-01-31 → 2026-03-31

📁 Files saved:
  ✅ d:\DS108\ds108\data\processed\Data_DS108_processed.csv

🔑 Data Leakage Prevention Summary:
  ✓ shift(-1) dùng để tạo target, KHÔNG dùng trong features
  ✓ Mọi shift(n)/rolling dùng groupby(symbol) — không mix company
  ✓ rolling() dùng shift(1) trước → không dùng giá trị hiện tại
  ✓ [Section 2] resolve_business_duplicates() giữ bản ít NaN nhất, mới nhất
  ✓ [Section 5] SignLogTransformer tất định → an toàn trước split, zero leakage
  ✓ [Section 7] 